In [18]:
from sentence_transformers import SentenceTransformer
import torch
# Load embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("google/embeddinggemma-300m", device=device)
print(f"Using device: {device}")

Using device: cuda


In [19]:
import os
dir_path = os.getcwd()
print("The directory of this script is:", dir_path)
root_path = os.path.dirname(dir_path)
print("The root directory is:", root_path)

The directory of this script is: c:\Users\HP\Desktop\Projects\NodeRAG\testing
The root directory is: c:\Users\HP\Desktop\Projects\NodeRAG


In [20]:
import pandas as pd
medical_questions = pd.read_parquet(f"{root_path}\GraphRAG-Benchmark\Datasets\Questions\medical_questions.parquet")
medical_questions = medical_questions[medical_questions.question_type == 'Complex Reasoning']
medical_questions

,id,source,question,answer,question_type,evidence,evidence_relations
1098,Medical-604c9d44,Medical,Why is a patient with fair skin and a history ...,Because both fair skin and immune suppression ...,Complex Reasoning,Fair skin is an independent risk factor for ba...,"Fair skin, light hair, and light eye color inc..."
1099,Medical-df366063,Medical,If a patient presents with a shiny bump on the...,"A medical and family history, physical and ski...",Complex Reasoning,A medical and family history should be perform...,BCC presents as shiny bumps; BCC most commonly...
1100,Medical-0a90f1f2,Medical,How does the management of recurrent basal cel...,Management of recurrence depends on risk and r...,Complex Reasoning,Management of recurrent basal cell carcinoma d...,Recurrence refers to BCC returning to the orig...
1101,Medical-831bb1e6,Medical,Which combination of risk factors should promp...,"A combination of older age, fair skin, and fam...",Complex Reasoning,Older age is a risk factor for basal cell carc...,Older age is associated with higher risk of BC...
1102,Medical-01e3a9ea,Medical,Why might a patient with a shiny bump and a hi...,Because radiation therapy is a risk factor for...,Complex Reasoning,A shiny bump can be a symptom of basal cell ca...,Radiation therapy is a risk factor for BCC; BC...
...,...,...,...,...,...,...,...
1602,Medical-1eda4787,Medical,Which patients with breast cancer should recei...,All patients of reproductive potential should ...,Complex Reasoning,All patients of reproductive potential should ...,Fertility preservation should be discussed bef...
1603,Medical-24f2e8ea,Medical,How does the presence of HER2 positivity alter...,HER2 positivity adds HER2-targeted therapy to ...,Complex Reasoning,HER2 positivity adds HER2-targeted therapy to ...,HER2-targeted therapy is used for HER2-positiv...
1604,Medical-d1a87cb4,Medical,What combination of clinical findings and diag...,Symptoms like bone pain or neurological sympto...,Complex Reasoning,Symptoms like bone pain or neurological sympto...,Metastatic breast cancer (MBC) is breast cance...
1605,Medical-311d66d6,Medical,Which risk factors should be evaluated in a pa...,"Family history, BRCA1/2 mutations, and assigne...",Complex Reasoning,Family history should be evaluated as a risk f...,"Risk factors include family history, BRCA muta..."


In [21]:
question_ids = medical_questions['id'].to_list()
question_ids

['Medical-604c9d44',
 'Medical-df366063',
 'Medical-0a90f1f2',
 'Medical-831bb1e6',
 'Medical-01e3a9ea',
 'Medical-c69c2732',
 'Medical-f034d77d',
 'Medical-1eab8fa2',
 'Medical-14016ce3',
 'Medical-5e530443',
 'Medical-c8dd3e29',
 'Medical-a9a52704',
 'Medical-346b12e8',
 'Medical-8ff5dd5a',
 'Medical-322bb52d',
 'Medical-35fee661',
 'Medical-f473bcb0',
 'Medical-f3af6f24',
 'Medical-ade4b34a',
 'Medical-ade95016',
 'Medical-8435c9f6',
 'Medical-23e6b6df',
 'Medical-4744dfa7',
 'Medical-59ea36ca',
 'Medical-4124f00f',
 'Medical-ed26b623',
 'Medical-fbc9e7ec',
 'Medical-9a34e14c',
 'Medical-bfe2b31f',
 'Medical-b6fbf092',
 'Medical-55898188',
 'Medical-8b99b755',
 'Medical-e0f29bbd',
 'Medical-55f0c0e2',
 'Medical-8407a0ab',
 'Medical-7b4429ac',
 'Medical-6d12f4ff',
 'Medical-baed6d13',
 'Medical-eb7624ca',
 'Medical-0a6379ff',
 'Medical-462ecc23',
 'Medical-2782702e',
 'Medical-f978cb28',
 'Medical-4f5a8531',
 'Medical-db5aa2e3',
 'Medical-193a4b7f',
 'Medical-e9ac42f1',
 'Medical-788

In [22]:
import sys
sys.path.append(root_path)
from graphs.Node import Node
from answering.get_context import get_context
from answering.answer_prompt import answer_prompt

In [23]:
#Load Data
import pickle
import json
import faiss
import pandas as pd
import numpy as np
#Graph - Node dict
with open(f"{root_path}/graphs/data/graphs/G6_entity_and_chunk_resolution_graph.pkl", "rb") as f:
    medical_graph = pickle.load(f)

#Embeddings
hnsw = faiss.read_index(f"{root_path}/graphs/data/embedding/medical_index.faiss")
with open(f"{root_path}/graphs/data/embedding/medical_ids.json", "r") as f:
    medical_embedding_ids = json.load(f)
num_vectors = hnsw.ntotal
dimension = hnsw.d
embeddings = np.zeros((num_vectors, dimension), dtype='float32')
for i in range(num_vectors):
    embeddings[i] = hnsw.reconstruct(i)

#Entities
with open(f"{root_path}/graphs/data/nodes/entity/medical_entities.pkl", "rb") as f:
    medical_entities = pickle.load(f)
medical_overview = pd.read_parquet(f"{root_path}/graphs/data/nodes/community/medical_communities_overview.parquet")

In [24]:
#LLM api call
from google import genai
with open(f"{root_path}/API_KEY.txt", "r", encoding="utf-8") as f:
    API_KEY = f.read()

def call_gemini(text):
    client = genai.Client(api_key = API_KEY)
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=text
    )
    return response

In [25]:
graph_context = {
    'graph': medical_graph,
    'entities': medical_entities,
    'overview': medical_overview
}

embedding_context = {
    'model': model,
    'index': hnsw,
    'ids': medical_embedding_ids,
}

In [26]:
#Case 1: No graph reasoning

try:
    normal_rag_answer = pd.read_parquet("data/normal_rag_answer.parquet")
except:
    normal_rag_answer = medical_questions.copy(deep=True)
    normal_rag_answer[["LLM_answer", "LLM_context", "LLM_tokens"]] = None

normal_rag_answer

,id,source,question,answer,question_type,evidence,evidence_relations,LLM_answer,LLM_context,LLM_tokens
1098,Medical-604c9d44,Medical,Why is a patient with fair skin and a history ...,Because both fair skin and immune suppression ...,Complex Reasoning,Fair skin is an independent risk factor for ba...,"Fair skin, light hair, and light eye color inc...",None,None,None
1099,Medical-df366063,Medical,If a patient presents with a shiny bump on the...,"A medical and family history, physical and ski...",Complex Reasoning,A medical and family history should be perform...,BCC presents as shiny bumps; BCC most commonly...,None,None,None
1100,Medical-0a90f1f2,Medical,How does the management of recurrent basal cel...,Management of recurrence depends on risk and r...,Complex Reasoning,Management of recurrent basal cell carcinoma d...,Recurrence refers to BCC returning to the orig...,None,None,None
1101,Medical-831bb1e6,Medical,Which combination of risk factors should promp...,"A combination of older age, fair skin, and fam...",Complex Reasoning,Older age is a risk factor for basal cell carc...,Older age is associated with higher risk of BC...,None,None,None
1102,Medical-01e3a9ea,Medical,Why might a patient with a shiny bump and a hi...,Because radiation therapy is a risk factor for...,Complex Reasoning,A shiny bump can be a symptom of basal cell ca...,Radiation therapy is a risk factor for BCC; BC...,None,None,None
...,...,...,...,...,...,...,...,...,...,...
1602,Medical-1eda4787,Medical,Which patients with breast cancer should recei...,All patients of reproductive potential should ...,Complex Reasoning,All patients of reproductive potential should ...,Fertility preservation should be discussed bef...,None,None,None
1603,Medical-24f2e8ea,Medical,How does the presence of HER2 positivity alter...,HER2 positivity adds HER2-targeted therapy to ...,Complex Reasoning,HER2 positivity adds HER2-targeted therapy to ...,HER2-targeted therapy is used for HER2-positiv...,None,None,None
1604,Medical-d1a87cb4,Medical,What combination of clinical findings and diag...,Symptoms like bone pain or neurological sympto...,Complex Reasoning,Symptoms like bone pain or neurological sympto...,Metastatic breast cancer (MBC) is breast cance...,None,None,None
1605,Medical-311d66d6,Medical,Which risk factors should be evaluated in a pa...,"Family history, BRCA1/2 mutations, and assigne...",Complex Reasoning,Family history should be evaluated as a risk f...,"Risk factors include family history, BRCA muta...",None,None,None


In [27]:
LLM_answer_1 = (
    normal_rag_answer.loc[normal_rag_answer[["LLM_answer", "LLM_context", "LLM_tokens"]].notna().all(axis=1) &
        (normal_rag_answer[["LLM_answer", "LLM_context", "LLM_tokens"]] != "").all(axis=1)
        ].set_index("id")[["LLM_answer", "LLM_context", "LLM_tokens"]].to_dict(orient="index"))

LLM_answer_1

{}

In [28]:
#Context setup
ppr_context = {
    'k_ppr': 0,
    'alpha': 0.5,
    't':5
}
query_context = {
    'question': None,
    'k_embedding': 8,
    'ppr': ppr_context
}

In [ ]:
from tqdm import tqdm
import time
sep = "\n\n" + "-"*100 + "\n\n"
MAX_RETRIES = 20
SLEEP_SEC = 30

def get_answers_1():
    for qid in tqdm(question_ids):
        if qid in LLM_answer_1:
            continue
        row = medical_questions.loc[medical_questions["id"] == qid].iloc[0]
        question = row["question"]
        query_context["question"] = question
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                context = get_context(query_context,graph_context,embedding_context,API_KEY,debug=False)
                full_context = sep.join(context.values())
                prompt = answer_prompt(full_context, question)
                response = call_gemini(prompt)
                answer = response.text
                usage = response.usage_metadata
                #log
                LLM_answer_1[qid] = {
                    "LLM_answer": answer,
                    "LLM_context": full_context,
                    "LLM_tokens": usage.total_token_count,
                }
                break

            except Exception as e:
                code = e.error.code if hasattr(e, "error") else e
                print(f"[ERROR] qid={qid} failed after {attempt} retries: {code}")
                if attempt == MAX_RETRIES:
                    return
                time.sleep(SLEEP_SEC * attempt)  #backoff

get_answers_1()


  0%|          | 0/509 [00:05<?, ?it/s]


In [30]:
LLM_answer_1

{'Medical-604c9d44': {'LLM_answer': 'A patient with fair skin is at a higher risk for basal cell carcinoma because their skin has less melanin to filter harmful UV rays. Additionally, individuals with a history of organ transplant experience immune suppression, which further increases their risk of developing basal cell carcinoma.',
  'LLM_context': "Here are distinct categories of high-level information extracted from the text cluster, providing a well-rounded overview:\n\n---\n\n### 1. Common Skin Cancers: Prevalence and Curability\n\n**Description:** This cluster predominantly addresses Basal Cell Carcinoma (BCC) and Squamous Cell Carcinoma (CSCC), identifying them as the most common types of skin cancer, collectively known as nonmelanoma skin cancers. BCC is the most prevalent, with approximately 3 million annual diagnoses in the U.S., while CSCC is the second most common. Both are generally highly treatable and curable, especially when detected early, owing to their less aggressiv

In [31]:
llm_df_1 = (
    pd.DataFrame.from_dict(LLM_answer_1, orient="index")
      .reset_index()
      .rename(columns={"index": "id"})
)
llm_df_1

,id,LLM_answer,LLM_context,LLM_tokens
0,Medical-604c9d44,A patient with fair skin is at a higher risk f...,Here are distinct categories of high-level inf...,2727


In [32]:
print("min:", llm_df_1["LLM_tokens"].min())
print("max:", llm_df_1["LLM_tokens"].max())
print("avg:", llm_df_1["LLM_tokens"].mean())
print("sum:", llm_df_1["LLM_tokens"].sum())

min: 2727
max: 2727
avg: 2727.0
sum: 2727


In [33]:
normal_rag_answer = medical_questions.merge(
    llm_df_1,
    on="id",
    how="left"
)
normal_rag_answer

,id,source,question,answer,question_type,evidence,evidence_relations,LLM_answer,LLM_context,LLM_tokens
0,Medical-604c9d44,Medical,Why is a patient with fair skin and a history ...,Because both fair skin and immune suppression ...,Complex Reasoning,Fair skin is an independent risk factor for ba...,"Fair skin, light hair, and light eye color inc...",A patient with fair skin is at a higher risk f...,Here are distinct categories of high-level inf...,2727.0
1,Medical-df366063,Medical,If a patient presents with a shiny bump on the...,"A medical and family history, physical and ski...",Complex Reasoning,A medical and family history should be perform...,BCC presents as shiny bumps; BCC most commonly...,NaN,NaN,NaN
2,Medical-0a90f1f2,Medical,How does the management of recurrent basal cel...,Management of recurrence depends on risk and r...,Complex Reasoning,Management of recurrent basal cell carcinoma d...,Recurrence refers to BCC returning to the orig...,NaN,NaN,NaN
3,Medical-831bb1e6,Medical,Which combination of risk factors should promp...,"A combination of older age, fair skin, and fam...",Complex Reasoning,Older age is a risk factor for basal cell carc...,Older age is associated with higher risk of BC...,NaN,NaN,NaN
4,Medical-01e3a9ea,Medical,Why might a patient with a shiny bump and a hi...,Because radiation therapy is a risk factor for...,Complex Reasoning,A shiny bump can be a symptom of basal cell ca...,Radiation therapy is a risk factor for BCC; BC...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
504,Medical-1eda4787,Medical,Which patients with breast cancer should recei...,All patients of reproductive potential should ...,Complex Reasoning,All patients of reproductive potential should ...,Fertility preservation should be discussed bef...,NaN,NaN,NaN
505,Medical-24f2e8ea,Medical,How does the presence of HER2 positivity alter...,HER2 positivity adds HER2-targeted therapy to ...,Complex Reasoning,HER2 positivity adds HER2-targeted therapy to ...,HER2-targeted therapy is used for HER2-positiv...,NaN,NaN,NaN
506,Medical-d1a87cb4,Medical,What combination of clinical findings and diag...,Symptoms like bone pain or neurological sympto...,Complex Reasoning,Symptoms like bone pain or neurological sympto...,Metastatic breast cancer (MBC) is breast cance...,NaN,NaN,NaN
507,Medical-311d66d6,Medical,Which risk factors should be evaluated in a pa...,"Family history, BRCA1/2 mutations, and assigne...",Complex Reasoning,Family history should be evaluated as a risk f...,"Risk factors include family history, BRCA muta...",NaN,NaN,NaN


In [34]:
normal_rag_answer.to_parquet("data/normal_rag_answer.parquet")
normal_rag_answer_loaded = pd.read_parquet("data/normal_rag_answer.parquet")
normal_rag_answer_loaded

,id,source,question,answer,question_type,evidence,evidence_relations,LLM_answer,LLM_context,LLM_tokens
0,Medical-604c9d44,Medical,Why is a patient with fair skin and a history ...,Because both fair skin and immune suppression ...,Complex Reasoning,Fair skin is an independent risk factor for ba...,"Fair skin, light hair, and light eye color inc...",A patient with fair skin is at a higher risk f...,Here are distinct categories of high-level inf...,2727.0
1,Medical-df366063,Medical,If a patient presents with a shiny bump on the...,"A medical and family history, physical and ski...",Complex Reasoning,A medical and family history should be perform...,BCC presents as shiny bumps; BCC most commonly...,None,None,NaN
2,Medical-0a90f1f2,Medical,How does the management of recurrent basal cel...,Management of recurrence depends on risk and r...,Complex Reasoning,Management of recurrent basal cell carcinoma d...,Recurrence refers to BCC returning to the orig...,None,None,NaN
3,Medical-831bb1e6,Medical,Which combination of risk factors should promp...,"A combination of older age, fair skin, and fam...",Complex Reasoning,Older age is a risk factor for basal cell carc...,Older age is associated with higher risk of BC...,None,None,NaN
4,Medical-01e3a9ea,Medical,Why might a patient with a shiny bump and a hi...,Because radiation therapy is a risk factor for...,Complex Reasoning,A shiny bump can be a symptom of basal cell ca...,Radiation therapy is a risk factor for BCC; BC...,None,None,NaN
...,...,...,...,...,...,...,...,...,...,...
504,Medical-1eda4787,Medical,Which patients with breast cancer should recei...,All patients of reproductive potential should ...,Complex Reasoning,All patients of reproductive potential should ...,Fertility preservation should be discussed bef...,None,None,NaN
505,Medical-24f2e8ea,Medical,How does the presence of HER2 positivity alter...,HER2 positivity adds HER2-targeted therapy to ...,Complex Reasoning,HER2 positivity adds HER2-targeted therapy to ...,HER2-targeted therapy is used for HER2-positiv...,None,None,NaN
506,Medical-d1a87cb4,Medical,What combination of clinical findings and diag...,Symptoms like bone pain or neurological sympto...,Complex Reasoning,Symptoms like bone pain or neurological sympto...,Metastatic breast cancer (MBC) is breast cance...,None,None,NaN
507,Medical-311d66d6,Medical,Which risk factors should be evaluated in a pa...,"Family history, BRCA1/2 mutations, and assigne...",Complex Reasoning,Family history should be evaluated as a risk f...,"Risk factors include family history, BRCA muta...",None,None,NaN
